# Logos Baseline Debug Run

Minimal pretraining of a **Logos** transformer — small enough to run on a free Colab T4 or a laptop CPU. All current architecture pieces (KDA, SWA, CSA, HCA, Block Attention Residuals, MoE with bias balancing) are exercised. W&B logs offline so no API key is needed.

## 1. GPU / CPU check

In [ ]:
import torch, sys, os, multiprocessing

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    mem_gb = props.total_memory / 1e9
    device_str = f"{props.name} ({mem_gb:.1f} GB)"
    device = "cuda"
    print(f"GPU: {device_str}")
    !nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null || true
else:
    device = "cpu"
    print(f"No GPU — running on CPU ({multiprocessing.cpu_count()} cores)")
    print("Expect ~1-2 steps/sec. Use a T4 runtime for faster iteration.")

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {device}")

## 2. Install dependencies

In [ ]:
os.environ.setdefault('TORCHINDUCTOR_COMPILE_THREADS', str(multiprocessing.cpu_count()))

# Colab ships with a sufficiently recent torch. Only upgrade if needed.
need_upgrade = tuple(int(x) for x in torch.__version__.split('+')[0].split('.')[:2]) < (2, 10)
if need_upgrade and torch.cuda.is_available():
    print(f"torch {torch.__version__} is old, upgrading...")
    !pip install -q --upgrade torch --index-url https://download.pytorch.org/whl/cu130
    print("Restart the runtime (Runtime → Restart) and re-run from cell 1.")
    import sys; sys.exit(0)
else:
    print(f"torch {torch.__version__} OK.")

!pip install -q -U datasets transformers tiktoken einops tqdm wandb

## 3. Clone the Logos repo

Replace the URL with your fork if you have one.

In [ ]:
%cd /content/
import pathlib, subprocess

REPO_URL  = 'https://github.com/Rorical/Logos.git'
REPO_DIR  = '/content/Logos'

if pathlib.Path(REPO_DIR).exists():
    print("Repo exists — fetching + hard-resetting to origin/master")
    subprocess.check_call(['git', '-C', REPO_DIR, 'fetch', '--all', '--prune'])
    subprocess.check_call(['git', '-C', REPO_DIR, 'reset', '--hard', 'origin/master'])
else:
    subprocess.check_call(['git', 'clone', REPO_URL, REPO_DIR])

print('HEAD:')
subprocess.check_call(['git', '-C', REPO_DIR, 'log', '--oneline', '-1'])
%cd /content/Logos

## 4. Checkpoint directory (local, no Drive mount)

In [ ]:
import pathlib

CHECKPOINT_DIR = '/content/logos-run'
pathlib.Path(CHECKPOINT_DIR).mkdir(parents=True, exist_ok=True)

INDUCTOR_CACHE = '/content/inductor_cache'
TRITON_CACHE   = '/content/triton_cache'
pathlib.Path(INDUCTOR_CACHE).mkdir(parents=True, exist_ok=True)
pathlib.Path(TRITON_CACHE).mkdir(parents=True, exist_ok=True)
os.environ['TORCHINDUCTOR_CACHE_DIR'] = INDUCTOR_CACHE
os.environ['TRITON_CACHE_DIR'] = TRITON_CACHE

print(f"Checkpoints:    {CHECKPOINT_DIR}")
print(f"Inductor cache: {INDUCTOR_CACHE}")
print(f"Triton cache:   {TRITON_CACHE}")

## 5. Imports + API compatibility

In [ ]:
sys.path.insert(0, '/content/Logos')
from scripts.train import build_arg_parser, main
from models.logos import LogosConfig
import scripts.train as st

required = ['build_arg_parser', 'main', 'PackedStream',
            'build_streaming_loaders', 'run_step_training',
            'split_param_groups', 'build_optimizer_and_scheduler']
missing = [r for r in required if not hasattr(st, r)]
if missing:
    raise RuntimeError(f"Missing APIs: {missing}. Re-run the clone cell to fetch latest.")

print("API compatibility OK.")

## 6. Build the training Namespace

Minimal config that exercises every component. All flags below are `scripts/train.py` arguments — no duplicated model/optimizer/loop code lives in this notebook.

In [ ]:
parser = build_arg_parser()

cli = [
    '--model', 'logos',

    # ---- Streaming (light shard) ----
    '--streaming',
    '--dataset',         'HuggingFaceFW/fineweb-edu',
    '--dataset-config',  'sample-10BT',
    '--val-docs',        '20',
    '--total-steps',     '200',
    '--batch-size',      '4',
    '--max-length',      '256',
    '--log-every',       '20',
    '--eval-every',      '100',
    '--save-every',      '200',
    '--sample-every',    '200',
    '--num-workers',     '2',
    '--prefetch-factor', '2',

    # ---- Architecture ----
    '--d-model',          '384',
    '--num-heads',        '6',
    '--head-dim',         '64',
    '--d-ff',             '1024',
    '--num-entry-layers', '1',
    '--num-body-layers',  '2',
    '--num-exit-layers',  '1',
    '--num-loops',        '2',

    # ---- KDA ----
    '--conv-size',  '4',
    '--chunk-size', '32',

    # ---- CSA / HCA ----
    '--entry-attn-pattern',  'hca',
    '--body-attn-pattern',   'hca,csa,kda,swa',
    '--exit-attn-pattern',   'csa',
    '--csa-compression',     '4',
    '--csa-top-k',           '8',
    '--csa-indexer-heads',   '4',
    '--csa-indexer-dim',     '32',
    '--hca-compression',     '64',
    '--compressed-head-dim', '64',

    # ---- SWA ----
    '--swa-window', '64',

    # ---- MoE ----
    '--use-moe',
    '--num-sparse-experts',  '4',
    '--num-shared-experts',  '1',
    '--top-k',               '2',
    '--expert-d-ff',         '64',
    '--moe-diversity-factor','0.5',
    '--bias-update-rate',    '0.05',

    # ---- Optimizer ----
    '--lr',            '1e-3',
    '--embed-lr',      '0.05',
    '--muon-lr',       '0.01',
    '--router-lr',     '1e-3',
    '--weight-decay',  '0.1',
    '--warmup-steps',  '100',
    '--grad-clip',     '1.0',
    '--scheduler',     'wsd',
    '--decay-frac',    '0.1',

    # ---- Diagnostics ----
    '--diagnostic-every', '50',

    # ---- Safety ----
    '--seed',    '42',
    '--no-save',
    '--wandb',
    '--wandb-project', 'logos-debug',
    '--wandb-mode',    'offline',
]

# On CPU, skip bf16 and compile. Reduce batch size.
if device == 'cpu':
    cli = [a for a in cli if a not in ('--bf16',)]
    for i, arg in enumerate(cli):
        if arg == '--batch-size':
            cli[i + 1] = '2'
            break

args = parser.parse_args(cli)
args.save_dir = CHECKPOINT_DIR

print("Configuration:")
for k in sorted(vars(args)):
    v = vars(args)[k]
    if not k.startswith('_'):
        print(f"  {k:30s} = {v}")

tokens_per_step = args.batch_size * args.max_length * (2 if torch.cuda.device_count() > 1 else 1)
print(f"\nTokens/step: {tokens_per_step:,}")
if args.total_steps:
    rate = 6 if device == 'cuda' else 1.5
    minutes = args.total_steps / rate / 60
    print(f"Total steps: {args.total_steps:,}")
    print(f"Est. wall time: ~{minutes:.0f} min")

## 7. Train

Single call. `main(args)` handles everything: tokenizer, streaming data, model build, four-way Muon/AdamW split, WSD scheduler, step-bounded loop with diagnostics, eval, and sampling.

In [ ]:
main(args)

## 8. Plot the loss curve

In [ ]:
import json, math, matplotlib.pyplot as plt
from pathlib import Path

hist_path = Path(CHECKPOINT_DIR) / 'history.json'
if hist_path.exists():
    with hist_path.open() as f:
        hist = json.load(f)

    events = hist['history']
    val_pts  = [(h['step'], h['val']['loss']) for h in events if h.get('val') and h['val']['loss'] != float('inf')]

    plt.figure(figsize=(10, 5))
    if val_pts:
        steps, losses = zip(*val_pts)
        ppls = [math.exp(min(l, 20)) for l in losses]
        plt.plot(steps, losses, 'o-', label='val loss', linewidth=2)
        ax2 = plt.twinx()
        ax2.plot(steps, ppls, 's--', color='orange', label='val ppl', alpha=0.7)
        ax2.set_ylabel('perplexity', color='orange')
        ax2.legend(loc='upper right')

    plt.xlabel('step')
    plt.ylabel('loss')
    plt.legend(loc='upper left')
    plt.grid(alpha=0.3)
    plt.title('Logos baseline debug run')
    plt.show()
    
    final = events[-1]
    print(f"Final step: {final['step']}")
    if final.get('val'):
        print(f"Final val loss: {final['val']['loss']:.4f}  (ppl {final['val']['ppl']:.2f})")
else:
    print(f"No history.json found at {hist_path} — training may not have completed.")

## 9. (Optional) Generate text from the model

In [ ]:
ckpt_path = Path(CHECKPOINT_DIR) / 'checkpoint_final.pt'
if not ckpt_path.exists():
    ckpts = sorted(Path(CHECKPOINT_DIR).glob('checkpoint_epoch_*.pt'))
    if ckpts:
        ckpt_path = ckpts[-1]
        print(f"Using {ckpt_path.name}")
    else:
        print("No checkpoint found — --no-save was used.")
        ckpt_path = None

if ckpt_path:
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)

    from models.logos import LogosConfig, LogosTransformer
    from utils.tokenizer import TiktokenTokenizer
    from torch.optim.swa_utils import AveragedModel, get_ema_multi_avg_fn

    tok = TiktokenTokenizer(args.tiktoken_encoding)
    cfg_kwargs = {k: v for k, v in vars(args).items() if k in LogosConfig.__dataclass_fields__}
    cfg = LogosConfig(**{**cfg_kwargs, 'vocab_size': tok.vocab_size})

    model = LogosTransformer(cfg).to(device)
    model.load_state_dict(ckpt['model_state_dict'])

    if ckpt.get('ema_state_dict'):
        ema = AveragedModel(model, multi_avg_fn=get_ema_multi_avg_fn(0.999), use_buffers=True).to(device)
        ema.load_state_dict(ckpt['ema_state_dict'])
        target = ema.module
        print('Using EMA weights')
    else:
        target = model
        print('Using live weights')

    prompt = 'The history of artificial intelligence began'
    ids = torch.tensor([tok.encode(prompt)], dtype=torch.long, device=device)
    target.eval()
    with torch.no_grad():
        amp_ctx = torch.autocast(device_type=device, dtype=torch.bfloat16) if device == 'cuda' else torch.inference_mode()
        with amp_ctx:
            out = target.generate(ids, max_new_tokens=100, temperature=0.7, top_k=40)
    print(tok.decode(out[0].tolist()))